In [98]:
##### Cleans FAO value added data
# aligns with core ISO codes and reformats data

import os
import pandas as pd

In [99]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import data
value_added = pd.read_csv(f"{cd}/Data/Raw/FAO_value_added/value_added.csv")

country_codes = pd.read_csv(
    f"{cd}/Data/Correspondence_tables/country_names.csv",
    encoding="cp1252"
)

# Set save path
save_path = f"{cd}/Data/Clean/Capital_stock/FAO_value_added.csv"


In [100]:
##### Clean data

# drop unnecessary columns
columns_to_keep = ['Area Code (M49)', 'Item', 'Year', 'Value']
value_added = value_added[columns_to_keep]

# rename columns
value_added = value_added.rename(columns={
    'Area Code (M49)': 'FAO_code',
    'Item': 'Sector',
    'Year': 'Year',
    'Value': 'Values'
})

# split agriculture and aggregate sectors into seperate columns
value_added_all = value_added[value_added['Sector'] == 'Value Added (Agriculture, Forestry and Fishing)']
value_added_ag = value_added[value_added['Sector'] == 'Value Added (Agriculture)']

value_added = value_added_all.merge(value_added_ag, on=['FAO_code', 'Year'])

# calcluate ag share of value added 
value_added['ag_share_value_added'] = (value_added['Values_y'] / value_added['Values_x']).clip(upper=1)

# drop unnecessary columns
columns_to_keep = ['FAO_code', 'Year', 'ag_share_value_added']
value_added = value_added[columns_to_keep]

# reformat to wide format
value_added_wide = value_added.pivot(
    index='FAO_code',
    columns='Year',
    values='ag_share_value_added'
).reset_index()

# add units  
value_added_wide['Units'] = 'Ag share of ag, forestry, and fishing value added'

# replace FAO_code 156 for 159
value_added_wide['FAO_code'] = value_added_wide['FAO_code'].replace({156: 159})

# merge with country codes to get ISO3 codes
value_added_wide = value_added_wide.merge(
    country_codes[['FAO_code', 'ISO3', 'Country_Name']],
    on='FAO_code',
    how='left'
)

# reorder columns
cols = ['ISO3', 'Country_Name', 'Units'] + [col for col in value_added_wide.columns if col not in ['ISO3', 'Country_Name', 'Units']]
value_added_wide = value_added_wide[cols]
value_added_wide = value_added_wide.drop(columns=['FAO_code'])


In [101]:
# Save
value_added_wide.to_csv(save_path, index=False)